In [ ]:
import numpy as np
import torch
in_data = np.load("/localdata/tomfre/FNO_data_Stokes/stokes_input.npy")
out_data = np.load("/localdata/tomfre/FNO_data_Stokes/stokes_output.npy")

In [ ]:
in_data = torch.load("/localdata/tomfre/FNO_data_Stokes/stokes_input.pt")
out_data = torch.load("/localdata/tomfre/FNO_data_Stokes/stokes_output.pt")

in_data.shape, out_data.shape

In [ ]:
data = torch.cat((in_data, out_data), dim=-1)
data.shape

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv(
    "/home/tomfre/Desktop/pioneer-backend/examples/pca_tuner_stokes/pytorch_trainer0.csv"
)

df_sorted = df.sort_values("MSEConstraint")
top_df = df_sorted.head(20)  # show best 50

fig = px.parallel_coordinates(
    top_df,
    color="MSEConstraint",
    color_continuous_scale=px.colors.sequential.Viridis
)
fig.show()

In [ ]:
px.density_heatmap(
    df,
    x="TorchPCANN_PCA components input",
    y="TorchPCANN_PCA components output",
    z="MSEConstraint",
)

In [ ]:
import multiprocessing as mp
import torch
import h5py
import pioneer

with h5py.File("/localdata/tomfre/darcy.hdf5", "r") as f:
    out_data = f["tensor"][:]  # type: ignore
    in_data = f["nu"][:]  # type: ignore

    in_data = torch.tensor(in_data, dtype=torch.float32).unsqueeze(-1)
    out_data = torch.tensor(out_data, dtype=torch.float32).permute(0, 2, 3, 1)

data = torch.cat((in_data, out_data), dim=-1)

In [ ]:
param = pioneer.optim.CategoricalHyperparameter(
            [(16, 16), (2, 2), (4, 4), (8, 8), (32, 32)]
        )

In [ ]:
import pioneer

spatial = pioneer.config.SpatialAxis(size=100)
feature = pioneer.config.FeatureAxis(size=10)
batch = pioneer.config.BatchAxis()

config_ellipsis = pioneer.config.DataConfiguration(float, [..., batch, feature], feature)
assert -2 == config_ellipsis.batch_axis_idx

In [ ]:
import os

gpu_visible = os.environ.get("CUDA_VISIBLE_DEVICES", "")
print("GPUs visible:", gpu_visible)

In [7]:
import pioneer

analyzer = pioneer.visualization.TuningAnalyzer(
    "/home/tomfre/Desktop/pioneer-backend/data_generation/dam/results/case0042"
)
#analyzer.print_tuning_overview()
analyzer.summary_statistics()

------ Summary Statistics ------

Tunable parameters:
TorchFCN_Hidden Neurons: unique values = 7
TorchFCN_Activation Function: unique values = 2
TorchFCN_Hidden Layers: unique values = 6
InverseNormalization_Normalization Active: unique values = 2

Metrics:
       MSEConstraint
count     160.000000
mean        0.194704
std         0.233760
min         0.021317
25%         0.047751
50%         0.076505
75%         0.228660
max         0.966454


In [8]:
correlation_matrix = analyzer.parameter_metric_correlation()

Correlation of parameters vs metrics:
                                          MSEConstraint
TorchFCN_Hidden Neurons                        0.562878
TorchFCN_Activation Function                  -0.011937
TorchFCN_Hidden Layers                         0.272937
InverseNormalization_Normalization Active      0.113524


In [9]:
analyzer.plot_heatmap("TorchFCN_Hidden Neurons", "TorchFCN_Hidden Layers", "MSEConstraint")

In [4]:
analyzer.plot_scatter("TorchFCN_Hidden Neurons", "MSEConstraint")

In [10]:
analyzer.plot_parallel_coordinates(top_k=20)